<a href="https://colab.research.google.com/github/nurfnick/Mathematical_Musings/blob/main/LetterGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import pandas as pd
import re
import os
from pathlib import Path

# ------------------------------------------------
# FILE LOCATIONS
# ------------------------------------------------

excel_file = "PaypalDonations2026Match.xlsx"
template_file = "template.tex"

output_folder = Path("Donation_Letters")
output_folder.mkdir(exist_ok=True)

# Set to True if you have pdflatex installed
COMPILE_PDF = False


# ------------------------------------------------
# LATEX FORMATTING
# ------------------------------------------------

def latex_escape(value):
    """Escape special LaTeX characters in donor information."""

    value = str(value)

    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    for char, replacement in replacements.items():
        value = value.replace(char, replacement)

    return value


def replace_command(template, command, value):
    """Replace the contents of a LaTeX newcommand."""

    pattern = (
        r"(\\newcommand\{\\"
        + re.escape(command)
        + r"\}\{).*?(\})"
    )

    return re.sub(
        pattern,
        lambda m: m.group(1) + value + m.group(2),
        template,
        count=1,
        flags=re.DOTALL
    )


# ------------------------------------------------
# READ EXCEL FILE AND LATEX TEMPLATE
# ------------------------------------------------

df = pd.read_excel(excel_file)

with open(template_file, "r", encoding="utf-8") as file:
    original_template = file.read()


# ------------------------------------------------
# GENERATE A LETTER FOR EACH DONOR
# ------------------------------------------------

for index, row in df.iterrows():

    # Skip rows without a donor name
    if pd.isna(row["Name"]):
        continue

    donor = latex_escape(row["Name"])
    address = latex_escape(row["Address"])
    city = latex_escape(row["City"])

    amount = float(row["Amount"])
    amount = f"{amount:,.2f}"

    # Start with a fresh copy of the template
    letter = original_template

    # Replace donor-specific LaTeX commands
    letter = replace_command(letter, "donor", donor)
    letter = replace_command(letter, "adress", address)
    letter = replace_command(letter, "town", city)
    letter = replace_command(letter, "amount", amount)

    # Insert the mailing address in the letter environment
    mailing_address = (
        address + r" \\" + "\n" +
        city
    )

    letter = re.sub(
        r"(\\begin\{letter\}\{).*?(\})",
        lambda m: m.group(1) + mailing_address + m.group(2),
        letter,
        count=1,
        flags=re.DOTALL
    )

    # Create a safe filename
    filename = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(row["Name"])
    ).strip("_")

    output_file = output_folder / f"{filename}.tex"

    # Save the LaTeX file
    with open(output_file, "w", encoding="utf-8") as file:
        file.write(letter)

    print(f"Created: {output_file}")

print("\nAll donation letters have been generated!")


# ------------------------------------------------
# OPTIONAL: COMPILE LETTERS INTO PDF FILES
# ------------------------------------------------

if COMPILE_PDF:

    import subprocess
    import shutil

    for tex_file in output_folder.glob("*.tex"):

        # Copy required graphics into the output folder
        source_dir = Path(template_file).resolve().parent

        for asset in [
            "miz-rugby.png",
            "signature.pdf",
            "signature.png",
            "signature.jpg"
        ]:
            source = source_dir / asset
            if source.exists():
                shutil.copy2(source, output_folder / asset)

        subprocess.run(
            [
                "pdflatex",
                "-interaction=nonstopmode",
                tex_file.name
            ],
            cwd=output_folder,
            check=True
        )

        print(f"Compiled PDF: {tex_file.stem}.pdf")


Created: Donation_Letters/Kevin_Baniak.tex
Created: Donation_Letters/Edward_Sweeney.tex
Created: Donation_Letters/Nick_Klaus.tex
Created: Donation_Letters/Cherryl_Dillon.tex
Created: Donation_Letters/Ellen_Ermeling.tex
Created: Donation_Letters/Matt_Brotemarkle.tex
Created: Donation_Letters/John_McGuire.tex
Created: Donation_Letters/Nathan_Mlady.tex
Created: Donation_Letters/Nicholas_Allen.tex
Created: Donation_Letters/John_Tarpoff.tex
Created: Donation_Letters/John_BuchHolz.tex
Created: Donation_Letters/Ross_Heller.tex
Created: Donation_Letters/Tyler_Lasley.tex
Created: Donation_Letters/W_Stemme_Farms_LLC.tex
Created: Donation_Letters/Cash.tex
Created: Donation_Letters/William_Mcdonald.tex
Created: Donation_Letters/Nicholas_Jacob.tex
Created: Donation_Letters/James_T_Dierker.tex
Created: Donation_Letters/Ross_Heller.tex

All donation letters have been generated!
